# Detection Pipeline (Steps 10a/b/c)

Auto-labels images with Grounding DINO and writes a YOLO-format detection dataset.

**Prerequisite:** run `data-preparation.ipynb` first (`metadata.csv` must exist).

**Outputs:**
- `data/processed/auto_labels.json` (cached bbox predictions)
- `data/processed/detection_data/{images,labels}/{train,val,test}/...`
- `data/processed/review_list.tsv` (only images needing manual review)


In [ ]:
import json
import shutil
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


def safe(name):
    """Filesystem-friendly name: replace spaces and path separators."""
    return str(name).replace(" ", "_").replace("/", "_").replace(":", "_")


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

PROCESSED_DIR = Path("data/processed")
DETECTION_DIR = PROCESSED_DIR / "detection_data"
DETECTION_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: must match data-preparation.ipynb for class-id consistency.
TARGET_SPECIES = [
    "Amanita muscaria",
    "Boletus edulis",
    "Cantharellus cibarius",
    "Pleurotus ostreatus",
    "Agaricus bisporus",
    "Laetiporus sulphureus",
    "Morchella esculenta",
    "Coprinus comatus",
]


In [ ]:
metadata = pd.read_csv(PROCESSED_DIR / "metadata.csv")
print(f"Loaded metadata: {len(metadata)} rows")


In [ ]:
%pip install -q transformers torch

import torch

# --- Configuration ---
MODEL_ID = "IDEA-Research/grounding-dino-tiny"
BOX_THRESHOLD = 0.25  # minimum confidence for a detection to be kept
TEXT_THRESHOLD = 0.25  # minimum text-image similarity
LABELS_CACHE = PROCESSED_DIR / "auto_labels.json"

# Species → class-id mapping (alphabetical for reproducibility)
SPECIES_TO_ID = {sp: i for i, sp in enumerate(sorted(TARGET_SPECIES))}
print(f"Species → class mapping ({len(SPECIES_TO_ID)} classes):")
for sp, cid in SPECIES_TO_ID.items():
    print(f"  {cid}: {sp}")

# --- Load model ---
model = None
processor = None
device = "cpu"
try:
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

    device = (
        "cuda"
        if torch.cuda.is_available()
        else ("mps" if torch.backends.mps.is_available() else "cpu")
    )
    print(f"Loading {MODEL_ID} on {device}...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(device)
    print("✓ Loaded successfully")
except Exception as e:
    print(f"⚠ Could not load Grounding DINO: {e}")
    print("  To enable auto-labeling: pip install transformers torch")
    print("  Falling back to center-crop bboxes (still better than full-frame).")


# --- Auto-label ---
def auto_label_image(image_path, species_name):
    """Run Grounding DINO on one image.
    Returns list of (xc, yc, w, h, confidence) in YOLO-normalized format.
    Returns empty list if the model is unavailable or inference fails."""
    if model is None:
        return []
    try:
        img = Image.open(image_path).convert("RGB")
        w, h = img.size
        # Use descriptive text prompt with species name for fine-grained detection
        text_prompt = f"a {species_name.lower()} mushroom"
        inputs = processor(images=img, text=text_prompt, return_tensors="pt")
        if device != "cpu":
            inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        # FIX: dict-style access; `inputs.input_ids` raises AttributeError because inputs is a dict
        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs["input_ids"],
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            target_sizes=[img.size[::-1]],
        )
        boxes = []
        for result in results:
            for box, score in zip(result["boxes"], result["scores"]):
                x0, y0, x1, y1 = box.tolist()
                # Convert absolute [x0,y0,x1,y1] → normalized [xc,yc,w,h]
                xc = ((x0 + x1) / 2) / w
                yc = ((y0 + y1) / 2) / h
                bw = (x1 - x0) / w
                bh = (y1 - y0) / h
                # Clamp to [0,1]
                xc, yc, bw, bh = (
                    max(0, min(1, xc)),
                    max(0, min(1, yc)),
                    max(0, min(1, bw)),
                    max(0, min(1, bh)),
                )
                boxes.append((xc, yc, bw, bh, round(score.item(), 4)))
        return boxes
    except (RuntimeError, ValueError, AttributeError) as e:
        print(f"  Auto-label failed for {image_path}: {e}")
        return []


# --- Run auto-labeling ---
labels_cache = {}
if LABELS_CACHE.exists():
    with open(LABELS_CACHE) as f:
        labels_cache = json.load(f)
    print(f"Loaded {len(labels_cache)} cached labels from {LABELS_CACHE}")

new_labels = 0
fallback_labels = 0
for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc="Auto-labeling"):
    key = row["image_path"]
    if key in labels_cache:
        continue
    species = row["canonical_species"]
    boxes = auto_label_image(key, species)
    if not boxes:
        # Fallback: center 80% crop — much better than old full-frame placeholder
        boxes = [(0.5, 0.5, 0.8, 0.8, 0.0)]
        fallback_labels += 1
    labels_cache[key] = {
        "class_id": SPECIES_TO_ID.get(species, 0),
        "species": species,
        "boxes": boxes,  # list of (xc, yc, w, h, conf)
        "source": "grounding-dino" if model is not None else "center-crop-fallback",
    }
    # FIX: this MUST be inside the for loop (it used to be at module scope,
    # producing a negative count when all detections fell back to center-crop).
    new_labels += 1

# Save cache
with open(LABELS_CACHE, "w") as f:
    json.dump(labels_cache, f, indent=2)

print(f"\nResults: {new_labels} new labels saved")
if model is not None:
    print(f"  Grounding DINO detections: {new_labels - fallback_labels}")
print(f"  Center-crop fallbacks: {fallback_labels}")
print(f"  Cache: {LABELS_CACHE}")


## 10b. Build the detection dataset (YOLO format — auto-labeled)

Creates a YOLO-style detection dataset with **per-species class labels** (8 classes, one per target species). Bounding boxes come from `auto_labels.json` (generated by Step 10a).

**Label format (YOLO):** `class_id x_center y_center width height`

**Note:** Images where Grounding DINO could not run use a center 80% crop fallback (confidence=0.0 in `auto_labels.json`).

Output structure (YOLO convention):

```text
data/processed/detection_data/
    images/{train,val,test}/<image>.jpg
    labels/{train,val,test}/<image>.txt
```

To train a YOLO detector:

```python
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='data/processed/detection_data/yolo.yaml', epochs=50, imgsz=640)
```


In [ ]:
# Load auto-generated labels from Step 10a

LABELS_CACHE = PROCESSED_DIR / "auto_labels.json"
auto_labels = {}
if LABELS_CACHE.exists():
    with open(LABELS_CACHE) as f:
        auto_labels = json.load(f)
    print(f"Loaded {len(auto_labels)} auto-generated labels")
else:
    print(
        "⚠ auto_labels.json not found. Run Step 10a first, or this cell will use center-crop fallbacks."
    )

# Species → class ID mapping (must match Step 10a)
SPECIES_TO_ID = {sp: i for i, sp in enumerate(sorted(TARGET_SPECIES))}

if len(metadata) == 0:
    print("No images to process. Run Steps 3-7 first.")
else:
    for split in ["train", "val", "test"]:
        (DETECTION_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (DETECTION_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    n_images, n_labels, n_real_bboxes = 0, 0, 0
    for i, (_, row) in enumerate(metadata.iterrows()):
        split = row.get("split")
        src = row.get("image_path")
        if pd.isna(split) or not src or not Path(src).exists():
            continue
        src = Path(src)
        stem = f"{i}_{src.stem}"
        img_dst = DETECTION_DIR / "images" / split / f"{stem}{src.suffix.lower()}"
        lbl_dst = DETECTION_DIR / "labels" / split / f"{stem}.txt"
        if not img_dst.exists():
            shutil.copy2(src, img_dst)
        n_images += 1

        # Get auto-generated bounding boxes
        label_data = auto_labels.get(row["image_path"], {})
        class_id = label_data.get("class_id", 0)
        boxes = label_data.get("boxes", [(0.5, 0.5, 0.8, 0.8, 0.0)])

        # Write YOLO-format label file
        with open(lbl_dst, "w") as f:
            for box in boxes:
                xc, yc, bw, bh = box[0], box[1], box[2], box[3]
                conf = box[4] if len(box) > 4 else 0.0
                # Standard YOLO format: class_id x_center y_center width height
                f.write(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
                n_labels += 1
                if conf > 0.0:
                    n_real_bboxes += 1

    # Write multi-class yolo.yaml
    species_names = {i: sp for sp, i in SPECIES_TO_ID.items()}
    names_yaml = "\n".join(f"  {i}: {sp}" for i, sp in sorted(species_names.items()))
    yolo_yaml = (
        f"path: {DETECTION_DIR.resolve()}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"test: images/test\n"
        f"nc: {len(SPECIES_TO_ID)}\n"
        f"names:\n"
        f"{names_yaml}\n"
    )
    (DETECTION_DIR / "yolo.yaml").write_text(yolo_yaml)

    print(f"Detection dataset at: {DETECTION_DIR.resolve()}")
    print(
        f"  {n_images} images, {n_labels} labels ({n_real_bboxes} real, {n_labels - n_real_bboxes} fallback)"
    )
    print(f"  {len(SPECIES_TO_ID)} classes in yolo.yaml")
    for split in ["train", "val", "test"]:
        n_img = sum(
            1 for f in (DETECTION_DIR / "images" / split).iterdir() if f.is_file()
        )
        n_lbl = sum(
            1 for f in (DETECTION_DIR / "labels" / split).iterdir() if f.is_file()
        )
        print(f"  {split}: {n_img} images, {n_lbl} labels")


## 10c. Manual correction workflow (optional)

The auto-generated bounding boxes from Grounding DINO are good but not perfect. This section helps you find and fix the worst predictions.

### Quick quality assessment

Run the cell below to see which images need attention:
- Images with **confidence < 0.3**: likely need manual correction
- Images with **no detections** (fallback center-crop): definitely need manual review

### Manual labeling tools

For the images that need correction, use one of these tools:

| Tool | Best for | Setup time |
|---|---|---|
| **[Roboflow](https://roboflow.com/)** | Cloud-based, collaborative, AI-assisted | 5 min |
| **[LabelImg](https://github.com/HumanSignal/labelImg)** | Local, simple, YOLO-native | 2 min |
| **[CVAT](https://www.cvat.ai/)** | Self-hosted, advanced features | 30 min |

### Workflow

1. Run the cell below to identify low-confidence images
2. Open those images in your labeling tool of choice
3. Draw/adjust bounding boxes for each mushroom
4. Export labels in YOLO format, overwriting the auto-generated ones in `data/processed/detection_data/labels/`
5. Re-run the YOLO training with the corrected dataset


In [ ]:
# Identify images that need manual review

LABELS_CACHE = PROCESSED_DIR / "auto_labels.json"
LOW_CONF_THRESHOLD = 0.3

if not LABELS_CACHE.exists():
    print(f"{LABELS_CACHE} not found. Run Step 10a first.")
else:
    with open(LABELS_CACHE) as f:
        labels = json.load(f)

    # Find low-confidence and fallback images
    low_conf = []
    fallback = []
    good = []
    for path, data in labels.items():
        boxes = data.get("boxes", [])
        if not boxes or boxes[0][4] == 0.0:
            fallback.append((path, data.get("species", "unknown")))
        elif any(b[4] < LOW_CONF_THRESHOLD for b in boxes):
            confs = [f"{b[4]:.2f}" for b in boxes]
            low_conf.append((path, data.get("species", "unknown"), confs))
        else:
            good.append(path)

    print(f"Quality report ({len(labels)} total images):")
    print(f"  ✅ Good (all bboxes ≥ {LOW_CONF_THRESHOLD}): {len(good)}")
    print(f"  🟡 Needs review (some bboxes < {LOW_CONF_THRESHOLD}): {len(low_conf)}")
    print(f"  🔴 Fallback (no detection, center-crop used): {len(fallback)}")

    if fallback:
        print(f"\n🔴 Fallback images to manually label first ({len(fallback)}):")
        for path, sp in fallback[:10]:
            print(f"  {path} ({sp})")
        if len(fallback) > 10:
            print(f"  ... and {len(fallback) - 10} more")

    if low_conf:
        print(f"\n🟡 Low-confidence images to review ({len(low_conf)}):")
        for path, sp, confs in low_conf[:10]:
            print(f"  {path} ({sp}) conf={confs}")
        if len(low_conf) > 10:
            print(f"  ... and {len(low_conf) - 10} more")

    # Save review list for external labeling tools
    review_list = PROCESSED_DIR / "review_list.tsv"
    with open(review_list, "w") as f:
        f.write("image_path\texpected_species\tstatus\n")
        for path, sp in fallback:
            f.write(f"{path}\t{sp}\tfallback_no_detection\n")
        for path, sp, confs in low_conf:
            f.write(f"{path}\t{sp}\tlow_confidence={','.join(confs)}\n")
    print(f"\nReview list saved to {review_list}")
    print("Open these images in LabelImg, Roboflow, or CVAT to correct the bboxes.")
